In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


In [2]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.compose import make_column_transformer, ColumnTransformer
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
df1 = df.copy()
test_df = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv')
test_df1 = test_df.copy()

In [4]:
df1.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Since there are limited features, we will use all the features

In [5]:
# We shall combine the test and train data into a single table and do modifications on the datasets

tt_df = pd.concat([df1, test_df1], axis=0).reset_index()
tt_df

,index,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500.0
1,1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500.0
2,2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500.0
3,3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000.0
4,4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,1454,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,6,2006,WD,Normal,NaN
2915,1455,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml,NaN
2916,1456,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml,NaN
2917,1457,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,NaN


In [6]:
# Passing the "SalePrice" into a seperate variable 'target' which will be used to fit the model i.e the value to be predicted
target = tt_df.pop('SalePrice')
target = target[0:len(df1)]
target

0       208500.0
1       181500.0
2       223500.0
3       140000.0
4       250000.0
          ...   
1455    175000.0
1456    210000.0
1457    266500.0
1458    142125.0
1459    147500.0
Name: SalePrice, Length: 1460, dtype: float64

In [7]:
# Dropping 'index' and 'Id' columns from table as they don't contributed to predictions
tt_df.drop(['index','Id'], axis=1, inplace=True)

In [8]:
# A simple function for seperating features into numerical and categorical lists
def num_cat_seperator(dataframe):
  categorical_cols = []
  numerical_cols = []
  for col in dataframe.columns:
    
    if (dataframe[col].dtypes == 'float64') | (dataframe[col].dtypes == 'int64') :
      numerical_cols.append(col)
    else:
      categorical_cols.append(col)

  print(f'categorical column lengh: {len(categorical_cols)}')
  print(f'numerical column lengh: {len(numerical_cols)}')
  return categorical_cols, numerical_cols

In [9]:
cat, num = num_cat_seperator(tt_df)

categorical column lengh: 43
numerical column lengh: 36


In [10]:
# If we go by data description, we can observe that certain categorical columns have value 'NA' which is not 
# a missing value, but rather that the feature lacks character. We will seperate out those in a list cat1 and remove
# those from 'cat' list.

cat1 = [
        'Alley',
        'BsmtQual',
        'BsmtCond',
        'BsmtExposure',
        'BsmtFinType1',
        'BsmtFinType2',
        'FireplaceQu',
        'GarageType',
        'GarageFinish',
        'GarageQual',
        'GarageCond',
        'PoolQC',
        'Fence',
        'MiscFeature',
        ]

for item in cat1:
  cat.remove(item)
len(cat)

29

In [11]:
# Seeing the number of 'na' values in cat1. Remember these are not missing values.
tt_df[cat1].isna().sum()

Alley           2721
BsmtQual          81
BsmtCond          82
BsmtExposure      82
BsmtFinType1      79
BsmtFinType2      80
FireplaceQu     1420
GarageType       157
GarageFinish     159
GarageQual       159
GarageCond       159
PoolQC          2909
Fence           2348
MiscFeature     2814
dtype: int64

In [12]:
# Filling those 'na' values with 'None' and checking 'na' vlues gin
tt_df[cat1] = tt_df[cat1].fillna('None')
tt_df[cat1].isna().sum()

Alley           0
BsmtQual        0
BsmtCond        0
BsmtExposure    0
BsmtFinType1    0
BsmtFinType2    0
FireplaceQu     0
GarageType      0
GarageFinish    0
GarageQual      0
GarageCond      0
PoolQC          0
Fence           0
MiscFeature     0
dtype: int64

In [13]:
# Checking 'na' values in cat list
tt_df[cat].isna().sum()

MSZoning          4
Street            0
LotShape          0
LandContour       0
Utilities         2
LotConfig         0
LandSlope         0
Neighborhood      0
Condition1        0
Condition2        0
BldgType          0
HouseStyle        0
RoofStyle         0
RoofMatl          0
Exterior1st       1
Exterior2nd       1
MasVnrType       24
ExterQual         0
ExterCond         0
Foundation        0
Heating           0
HeatingQC         0
CentralAir        0
Electrical        1
KitchenQual       1
Functional        2
PavedDrive        0
SaleType          1
SaleCondition     0
dtype: int64

In [14]:
# Imputing 'cat' values with median 
imputer = SimpleImputer(strategy='most_frequent')
tt_df[cat] = imputer.fit_transform(tt_df[cat])

# Rechecking 'na' values of 'cat' columns again
tt_df[cat].isna().sum()

MSZoning         0
Street           0
LotShape         0
LandContour      0
Utilities        0
LotConfig        0
LandSlope        0
Neighborhood     0
Condition1       0
Condition2       0
BldgType         0
HouseStyle       0
RoofStyle        0
RoofMatl         0
Exterior1st      0
Exterior2nd      0
MasVnrType       0
ExterQual        0
ExterCond        0
Foundation       0
Heating          0
HeatingQC        0
CentralAir       0
Electrical       0
KitchenQual      0
Functional       0
PavedDrive       0
SaleType         0
SaleCondition    0
dtype: int64

In [15]:
# Checking 'na' values of 'cat' columns
tt_df[num].isna().sum()

MSSubClass         0
LotFrontage      486
LotArea            0
OverallQual        0
OverallCond        0
YearBuilt          0
YearRemodAdd       0
MasVnrArea        23
BsmtFinSF1         1
BsmtFinSF2         1
BsmtUnfSF          1
TotalBsmtSF        1
1stFlrSF           0
2ndFlrSF           0
LowQualFinSF       0
GrLivArea          0
BsmtFullBath       2
BsmtHalfBath       2
FullBath           0
HalfBath           0
BedroomAbvGr       0
KitchenAbvGr       0
TotRmsAbvGrd       0
Fireplaces         0
GarageYrBlt      159
GarageCars         1
GarageArea         1
WoodDeckSF         0
OpenPorchSF        0
EnclosedPorch      0
3SsnPorch          0
ScreenPorch        0
PoolArea           0
MiscVal            0
MoSold             0
YrSold             0
dtype: int64

In [16]:
# Fill in the 'na' values with KNNImputer
imputer = KNNImputer(n_neighbors=4)
tt_df[num] = imputer.fit_transform(tt_df[num])

# Rechecking 'na' values of 'num' columns
tt_df[num].isna().sum()

MSSubClass       0
LotFrontage      0
LotArea          0
OverallQual      0
OverallCond      0
YearBuilt        0
YearRemodAdd     0
MasVnrArea       0
BsmtFinSF1       0
BsmtFinSF2       0
BsmtUnfSF        0
TotalBsmtSF      0
1stFlrSF         0
2ndFlrSF         0
LowQualFinSF     0
GrLivArea        0
BsmtFullBath     0
BsmtHalfBath     0
FullBath         0
HalfBath         0
BedroomAbvGr     0
KitchenAbvGr     0
TotRmsAbvGrd     0
Fireplaces       0
GarageYrBlt      0
GarageCars       0
GarageArea       0
WoodDeckSF       0
OpenPorchSF      0
EnclosedPorch    0
3SsnPorch        0
ScreenPorch      0
PoolArea         0
MiscVal          0
MoSold           0
YrSold           0
dtype: int64

In [17]:
# Making a new column for total bathrooms in a house
tt_df['Tot_Bathroom'] = tt_df['BsmtFullBath'] + 0.5 * tt_df['BsmtHalfBath'] + tt_df['FullBath'] + 0.5 * tt_df['HalfBath']

In [18]:
# Making a new column based on equal weightage to 'overallcondition' and 'OverallQuality'
tt_df['Avg_Qual_Cond'] = 0.5 * tt_df['OverallCond'] + 0.5 * tt_df['OverallQual']

In [19]:
# Making a new column for Square footage per room
tt_df['Room_sqfoot'] = tt_df['GrLivArea']/ (tt_df['TotRmsAbvGrd']+tt_df['FullBath']+
                                           tt_df['HalfBath']+tt_df['KitchenAbvGr'])

In [20]:
# Making a new column for above ground Square footage
tt_df['HiQualSF'] = tt_df['1stFlrSF'] + tt_df['2ndFlrSF']

In [21]:
# The 'MoSold' feature is cyclic in nature since it accounts for seasonality. Hence we can convert it accordingly
# to account for it using sin/cosine function.
tt_df['MoSold'] = -np.cos(0.5246*tt_df['MoSold']) # Converting month to cyclic using cos function

In [22]:
# Now we can see the skew of the numerical columns
tt_df[num].skew()

MSSubClass        1.376165
LotFrontage       1.369826
LotArea          12.829025
OverallQual       0.197212
OverallCond       0.570605
YearBuilt        -0.600114
YearRemodAdd     -0.451252
MasVnrArea        2.602007
BsmtFinSF1        1.426185
BsmtFinSF2        4.148206
BsmtUnfSF         0.919488
TotalBsmtSF       1.163360
1stFlrSF          1.470360
2ndFlrSF          0.862118
LowQualFinSF     12.094977
GrLivArea         1.270010
BsmtFullBath      0.625016
BsmtHalfBath      3.933616
FullBath          0.167692
HalfBath          0.694924
BedroomAbvGr      0.326492
KitchenAbvGr      4.304467
TotRmsAbvGrd      0.758757
Fireplaces        0.733872
GarageYrBlt      -0.319375
GarageCars       -0.218705
GarageArea        0.240972
WoodDeckSF        1.843380
OpenPorchSF       2.536417
EnclosedPorch     4.005950
3SsnPorch        11.381914
ScreenPorch       3.948723
PoolArea         16.907017
MiscVal          21.958480
MoSold           -0.753812
YrSold            0.132467
dtype: float64

In [23]:
# Collecting 'num' features with skew greater than 0.5 into a seperate list 'num_skew'
num_skew = tt_df[num].skew()[tt_df[num].skew() > 0.5].index.to_list()

In [24]:
# Transforming 'num_skew' to a near normal distribution using log(1+x) function
tt_df[num_skew] = np.log1p(tt_df[num_skew])

In [25]:
# Splitting the combined into train and test table seperately
tt_train = tt_df.iloc[0:len(df1), :]
tt_test = tt_df.iloc[len(df1):len(tt_df), :]

In [26]:
# Transforming the target variable also into near normal distribution using log(x) function since target does not have 0 values
target1 = np.log(target)

In [27]:
# Making a pipeline for processing 'num' and 'cat','cat1' features seperately

cat_preprocessing = make_pipeline(
    
    OneHotEncoder(handle_unknown="ignore", sparse=False),
)

# pipeline for numerical data
num_preprocessing = make_pipeline(StandardScaler())

# combine both pipeline using a columnTransformer
preprocessing = ColumnTransformer(
    [("num", num_preprocessing, num), ("cat", cat_preprocessing, cat+cat1)]
)

preprocessing

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('standardscaler',
                                                  StandardScaler())]),
                                 ['MSSubClass', 'LotFrontage', 'LotArea',
                                  'OverallQual', 'OverallCond', 'YearBuilt',
                                  'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
                                  'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                                  '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
                                  'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath',
                                  'FullBath', 'HalfBath...
                                 ['MSZoning', 'Street', 'LotShape',
                                  'LandContour', 'Utilities', 'LotConfig',
                                  'LandSlope', 'Neighborhood', 'Condition1',
                                  'Condition2', 'BldgType', 'HouseStyle',
          

In [28]:
# Making a Pipeline for evaluation of various models by fitting the data and targets and getting the best model using
# GridSearchCV

full_pipe = Pipeline(
    [
        ("preprocess", preprocessing),
        ("regressor", RandomForestRegressor()),
    ]
)

# model tunning with GridSearch
param_grid = {
    
    "regressor": [
        
        GradientBoostingRegressor(random_state=42),
        RandomForestRegressor(random_state=42),
        XGBRegressor(random_state=42),
        #Ridge(random_state=42)
    ],
}

grid = GridSearchCV(
    full_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True,
)

grid.fit(tt_train, target1)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('standardscaler',
                                                                                          StandardScaler())]),
                                                                         ['MSSubClass',
                                                                          'LotFrontage',
                                                                          'LotArea',
                                                                          'OverallQual',
                                                                          'OverallCond',
                                                                          'YearBuilt',
                                                                          'YearRemodAdd',
           

In [29]:
grid.best_params_

{'regressor': GradientBoostingRegressor(random_state=42)}

In [30]:
print(np.sqrt(-grid.best_score_))

0.1260178648258695


In [31]:
# We can see that GradientBoosting performed best. So we can try to tune its hyperparameters

full_pipe = Pipeline(
    [
        ("preprocess", preprocessing),
        ("regressor", GradientBoostingRegressor()),
    ]
)

param_grid = [
    {
        "regressor": [GradientBoostingRegressor(random_state=42)],
        "regressor__n_estimators": [100, 300, 500],
        "regressor__learning_rate": [0.1, 0.01],
        "regressor__max_depth": [2, 3, 5],
        "regressor__max_features": ["log2", "sqrt", "auto"],
    },
    
]


grid = GridSearchCV(
    full_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True,
)

grid.fit(tt_train, target1)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('standardscaler',
                                                                                          StandardScaler())]),
                                                                         ['MSSubClass',
                                                                          'LotFrontage',
                                                                          'LotArea',
                                                                          'OverallQual',
                                                                          'OverallCond',
                                                                          'YearBuilt',
                                                                          'YearRemodAdd',
           

In [32]:
print(np.sqrt(-grid.best_score_)) # We can see that score has improved with tuning

0.12122169357787976


In [33]:
# Using the grid's best regressor with tuned hyperparameter to make predictions

y_pred = grid.predict(tt_test)

# Transforming the predicted values back using exponential function ( We had transformed target values using log function)
y_pred = np.exp(y_pred)

In [34]:
# Converting predicted values into Dataframe

dataframe = {'Id':test_df1.Id, 'SalePrice':y_pred}
output = pd.DataFrame(dataframe)
output

,Id,SalePrice
0,1461,122910.640982
1,1462,165926.368346
2,1463,183030.804527
3,1464,199796.095874
4,1465,187980.447284
...,...,...
1454,2915,80688.185171
1455,2916,80219.535411
1456,2917,177151.230189
1457,2918,115117.554911


In [35]:
# Making Csv file out of output dataframe

output.to_csv('submission_GrBoost_featurengg', index = False)

In [36]:
## Let's make another prediction using CatBoostRegressor through pipeline and see if it performs better

full_pipe = Pipeline(
    [
        ("preprocess", preprocessing),
        ("regressor", RandomForestRegressor()),
    ]
)

# model tunning with GridSearch
param_grid = {
    
    "regressor": [
        CatBoostRegressor(random_state=42),
        GradientBoostingRegressor(random_state=42),
        RandomForestRegressor(random_state=42),
        XGBRegressor(random_state=42),
        #Ridge(random_state=42)
    ],
}

grid = GridSearchCV(
    full_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True,
)

grid.fit(tt_train, target1)


Learning rate set to 0.04196
0:	learn: 0.3912707	total: 63.3ms	remaining: 1m 3s
1:	learn: 0.3810908	total: 67.6ms	remaining: 33.7s
2:	learn: 0.3718392	total: 71.6ms	remaining: 23.8s
3:	learn: 0.3622998	total: 76.1ms	remaining: 19s
4:	learn: 0.3541861	total: 80.5ms	remaining: 16s
5:	learn: 0.3450529	total: 86.8ms	remaining: 14.4s
6:	learn: 0.3364253	total: 92.3ms	remaining: 13.1s
7:	learn: 0.3281244	total: 98.4ms	remaining: 12.2s
8:	learn: 0.3198792	total: 105ms	remaining: 11.6s
9:	learn: 0.3125751	total: 115ms	remaining: 11.3s
10:	learn: 0.3053970	total: 119ms	remaining: 10.7s
11:	learn: 0.2989446	total: 124ms	remaining: 10.2s
12:	learn: 0.2927556	total: 129ms	remaining: 9.79s
13:	learn: 0.2860005	total: 134ms	remaining: 9.46s
14:	learn: 0.2798686	total: 143ms	remaining: 9.37s
15:	learn: 0.2739877	total: 147ms	remaining: 9.07s
16:	learn: 0.2685041	total: 151ms	remaining: 8.72s
17:	learn: 0.2624777	total: 155ms	remaining: 8.43s
18:	learn: 0.2573457	total: 158ms	remaining: 8.15s
19:	lear

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('standardscaler',
                                                                                          StandardScaler())]),
                                                                         ['MSSubClass',
                                                                          'LotFrontage',
                                                                          'LotArea',
                                                                          'OverallQual',
                                                                          'OverallCond',
                                                                          'YearBuilt',
                                                                          'YearRemodAdd',
           

In [37]:
print(np.sqrt(-grid.best_score_))

0.11915202607198713


In [38]:
print(grid.best_params_)

{'regressor': <catboost.core.CatBoostRegressor object at 0x7f065fa96650>}


In [39]:
# Using CatBoostRegressor to predict through grid

y_pred1 = grid.predict(tt_test)

# Transforming predictions back using exponential function
y_pred1 = np.exp(y_pred1)

In [40]:
# Converting to Dataframe

dataframe1 = {'Id':test_df1.Id, 'SalePrice':y_pred1}
output1 = pd.DataFrame(dataframe1)
output1

,Id,SalePrice
0,1461,124724.637137
1,1462,157790.034726
2,1463,185338.819083
3,1464,193767.430304
4,1465,183724.995622
...,...,...
1454,2915,81606.686570
1455,2916,81590.608672
1456,2917,166379.654093
1457,2918,112756.193360


In [41]:
output1.to_csv('submission_CatBoost_featurengg',index = False)

In [42]:
# Combining Catboost and Gradient Boosting predictions with weightage of 0.6 & 0.4 respectively 
dataframe3 = {'Id':test_df1.Id, 'SalePrice':(0.6*y_pred1+0.4*y_pred)}
output2 = pd.DataFrame(dataframe3)
output2

,Id,SalePrice
0,1461,123999.038675
1,1462,161044.568174
2,1463,184415.613261
3,1464,196178.896532
4,1465,185427.176287
...,...,...
1454,2915,81239.286010
1455,2916,81042.179368
1456,2917,170688.284531
1457,2918,113700.737980


In [43]:
output2.to_csv('submission_CatBoost_GrBoost_featurengg',index = False)